## DoE Lite: Symbolic Differentiation

## General Function

In [17]:
import pyomo.environ as pyo

from pyomo.core.expr.calculus.diff_with_pyomo import reverse_sd
from pyomo.core.expr.visitor import identify_variables
from pyomo.common.collections import ComponentSet

import numpy as np

import pandas as pd

# For the ipopt executable
import idaes

def doe_lite(experiment, objective="A"):
    ''' A lightweight version of Pyomo.DoE.

    Arguments:
        experiment: an Experiment object
        objective:
            None: skip objective and return after converging Jacobian constraints
            "A": A-optimality, trace(FIM)
            "D": D-optimality, logdet(FIM)
    
    Returns:
        model: the model object
        jac: the Jacobian matrix
        fim: the Fisher information matrix
            
    '''

    ## Step 1: Create and converge the experiment model

    # Create a Pyomo model
    model = experiment.get_labeled_model()

    # Loop over the design variables, fix them
    for v in model.experiment_inputs:
        v.fix()

    print("Solving the square model with design variables fixed...")
    # Solve the model
    solver = pyo.SolverFactory('ipopt')
    results1 = solver.solve(model, tee=True)

    print("\nDone.\n\n")

    ## Step 2: Assemble the Jacobians via Symbolic Difference

    # Parameters
    # Create an empty component set
    param_set = ComponentSet()

    # Loop over the unknown model parameters
    for p in model.unknown_parameters.keys():
        param_set.add(p)

    # Assemble into a list
    
    param_list = list(param_set)

    # Measurements (outputs)
    # Create an empty component set
    output_set = ComponentSet()

    # Loop over the model outputs
    for o in model.experiment_outputs.keys():
        output_set.add(o)

    # Assemble into a list
    output_list = list(output_set)

    # Constraints and Variables
    # Create empty component sets
    con_set = ComponentSet() # These will be all constraints in the Pyomo model
    var_set = ComponentSet() # These will be all Pyomo variables in the Pyomo model

    # Loop over the active model constraints
    for c in model.component_data_objects(pyo.Constraint, descend_into=True, active=True):

        # Add constraint c to the constraint set
        con_set.add(c)

        # Loop over the variables in the constraint c
        # Note: changed this to include_fixed=True
        # Changed back to False to fix problem degree of freedom issues
        for v in identify_variables(c.body, include_fixed=False):
            # Add variable v to the variable set
            var_set.add(v)

    # recall that the parameters are fixed, so we did not
    # get them above. Let's add them now.
    for p in model.unknown_parameters.keys():
        var_set.add(p)

    # Assemble into lists
    con_list = list(con_set)
    var_list = list(var_set)

    # Assemble Jacobian 
    # Create an empty dictionary
    jac_dict = {}

    # Enumerate over the constraints
    for i,c in enumerate(con_list):
        # Check we only have equality constraints... otherwise this gets more complicated
        assert c.equality, "This function only works with equality constraints"
        
        # Perform symbolic differentiation
        der_map = reverse_sd(c.body)

        # Loop over the Pyomo variables, which includes 
        # parameters, measurements, control decisions
        for j,v in enumerate(var_list):
            # Check if the variable is in the derivative map
            if v in der_map:
                # Record the expression 
                deriv = der_map[v]
            else:
                # Otherwise, record 0
                deriv = 0
            # Save results in the Jacobian dictionary
            jac_dict[(i, j)] = deriv


    ## Build the constraints to compute the Jacobian

    # Create empty lists
    param_index = []
    model_var_index = []
    measurement_index = []
    model.me_included = pyo.Suffix(direction=pyo.Suffix.LOCAL)

    # Loop over the variables and determine which ones 
    # (and associated indices) are (a) parameters or 
    # (b) measurements
    for i, v in enumerate(var_set):
        # Check if the variable is a parameter
        if v in param_set:
            # If yes, record its index
            param_index.append(i)
        else:
            # Otherwise, it is a model variable
            model_var_index.append(i)

            # Check if the model variable is a measurement
            if v in output_set:
                # If yes, record its index
                measurement_index.append(i)
                model.me_included[v] = model.measurement_error[v]

    # Using the lists of indices to create Pyomo Sets
    model.param_index = pyo.Set(initialize=param_index)
    model.measurement_index = pyo.Set(initialize=measurement_index)
    model.constraint_index = pyo.Set(initialize=range(len(con_list)))
    model.var_index = pyo.Set(initialize=model_var_index)

    # Define a Pyomo variable for the Jacobian of the model variables 
    # (everything except parameters) with respect to the model parameters
    model.jac_variables_wrt_param = pyo.Var(model.var_index, model.param_index, initialize=0)

    # Calculate the Jacobian using the chain rule and total derivative definitions
    #
    # Prior comment:
    # This has an index mistake... jac_dict includes the parameters, but var_index skips them
    # We need to be more careful about the indices
    #
    # New reflection:
    # var_index is built from the indices in var_list, which includes the parameters
    # I think this is okay
    @model.Constraint(model.constraint_index, model.param_index)
    def jacobian_constraint(model, i, j):
        return jac_dict[(i,j)] == -sum(model.jac_variables_wrt_param[k,j] * jac_dict[(i,k)] for k in model.var_index)
    
    # Step 3: Solve the model with the Jacobian constraints, extract the Jacobian
    results2 = solver.solve(model, tee=True)

    def get_jac():
        jac = np.zeros((len(measurement_index), len(param_index)))

        for i,y in enumerate(model.measurement_index):
            for j,p in enumerate(model.param_index):
                # print(f"Jacobian of {var_list[y]} with respect to {var_list[p]}: {model.jac_variables_wrt_param[y,p].value}")
                jac[i,j] = model.jac_variables_wrt_param[y,p].value

        row_names = [str(var_list[y]) for y in model.measurement_index]
        col_names = [str(var_list[p]) for p in model.param_index]

        # print("jac = ", jac)
        # print("row_names = ", row_names)
        # print("col_names = ", col_names)

        jac_df = pd.DataFrame(jac, index=row_names, columns=col_names)

        return jac_df
    
    jac = get_jac()

    print("\nDone.\n\n")

    # Step 4: Compute the Fisher information matrix (FIM)

    print("Computing the Fisher information matrix (FIM)...")

    model.fim = pyo.Var(model.param_index, model.param_index, initialize=1)
    print("PRINTING LENGTHS\n\n\n\n\n")
    print(len(model.measurement_index))
    print(len([k for k in model.measurement_error]))
    print("\n\n\n\n\n")
    # I changed this to measurement index
    @model.Constraint(model.param_index, model.param_index)
    def fim_constraint(model, i, j):
        return model.fim[i,j] == sum((1 / model.measurement_error[val]) * model.jac_variables_wrt_param[model.measurement_index[ind + 1], i] * model.jac_variables_wrt_param[model.measurement_index[ind + 1], j] for ind, val in enumerate(model.me_included))

    # model.fim_constraint.pprint()
    model.T.pprint()
    results3 = solver.solve(model, tee=True)

    def get_fim():
        # Extract the FIM matrix
        fim = np.zeros((len(model.param_index), len(model.param_index)))
        for i, c in enumerate(model.param_index):
            for j, d in enumerate(model.param_index):
                fim[i, j] = model.fim[c, d].value
        
        # Grab the parameter names
        col_names = [str(var_list[c]) for c in model.param_index]

        # Store in a pandas dataframe
        fim_df = pd.DataFrame(fim, index=col_names, columns=col_names)

        return fim_df
    
    fim = get_fim()

    print("\nDone.\n\n")
    print(fim)

    if objective is None:
        return model, jac, fim
    else:

        print("Solving DoE optimization problem with {objective}-optimality objective")

        # Unfix the experiment design decisions
        for v in model.experiment_inputs:
            v.unfix()

    if objective == "A":
        @model.Objective(sense=pyo.maximize)
        def trace_fim(model):
            return sum(model.fim[i,i] for i in model.param_index)
    elif objective == "D":
        
        fim_array = fim.to_numpy()

        # Calculate the eigenvalues of the FIM matrix
        eig = np.linalg.eigvals(fim_array)

        # If the smallest eigenvalue is (practically) negative, add a diagonal matrix to make it positive definite
        small_number = 1e-10
        if min(eig) < small_number:
            fim_array = fim_array + np.eye(len(model.param_index)) * (
                small_number - min(eig)
            )

        # Compute the Cholesky decomposition of the FIM matrix
        L = np.linalg.cholesky(fim_array)

        model.L = pyo.Var(
                        model.param_index, model.param_index, initialize=0
                    )

        # loop over parameter name
        for i, c in enumerate(model.param_index):
            for j, d in enumerate(model.param_index):
                # fix the 0 half of L matrix to be 0.0
                if i < j:
                    model.L[c, d].fix(0.0)
                # Give LB to the diagonal entries
                elif i == j:
                    # Set the lower bound for the diagonal entries
                    # to be a small number
                    model.L[c, d].setlb(1E-10)
                            

        # Initialize the Cholesky matrix
        for i, c in enumerate(model.param_index):
            for j, d in enumerate(model.param_index):
                model.L[c, d].value = L[i, j]

        def cholesky_imp(m, c, d):
            """
            Calculate Cholesky L matrix using algebraic constraints
            """
            # If the row is greater than or equal to the column, we are in the
            # lower triangle region of the L and FIM matrices.
            # This region is where our equations are well-defined.
            if list(m.param_index).index(c) >= list(m.param_index).index(d):
                return m.fim[c, d] == sum(
                    m.L[c, m.param_index.at(k + 1)]
                    * m.L[d, m.param_index.at(k + 1)]
                    for k in range(list(m.param_index).index(d) + 1)
                )
            else:
                # This is the empty half of L above the diagonal
                return pyo.Constraint.Skip

        model.cholesky_cons = pyo.Constraint(
            model.param_index, model.param_index, rule=cholesky_imp
        )

        model.logdet_FIM = pyo.Objective(
            expr=2 * sum(pyo.log10(model.L[j, j]) for j in model.param_index),
            sense=pyo.maximize,
        )
    else:
        raise ValueError("Objective must be None, 'A' or 'D'")
        
    results4 = solver.solve(model, tee=True)
    print("\nDone.\n\n")

    jac = get_jac()
    fim = get_fim()

    return model, jac, fim


## Reactor Example

In [18]:
from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment # Load the example

# import json

# Copied from the json file in the example
# TODO: Use the json file instead of copying the data. Need to figure out how to load the file without hardcoding the path.
# Changed the control points ot see if FIM changes at initial point.
data_ex = {"CA0": 5.0, "CA_bounds": [1.0, 5.0], "CB0": 0.0, "CC0": 0.0, "t_range": [0.0, 1.0], "control_points": {0: 500, 0.125: 450, 0.25: 400, 0.375: 350, 0.5: 300, 0.625: 300, 0.75: 300, 0.875: 300, 1: 300}, "T_bounds": [300, 700], "A1": 84.79, "A2": 371.72, "E1": 7.78, "E2": 15.05}

experiment = ReactorExperiment(data=data_ex, nfe=10, ncp=3)

model, jac, fim = doe_lite(experiment, objective="D")

Solving the square model with design variables fixed...
Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for

In [19]:
print(jac)

                 A1            A2        E1            E2
CA[0.125] -0.019975 -2.211558e-77  0.427584  1.274850e-28
CA[0.25]  -0.016520 -6.497321e-77  0.404155  9.579593e-29
CA[0.375] -0.012855  3.715711e-30  0.338447 -3.186439e-27
CA[0.5]   -0.009621 -2.860354e-29  0.265361 -9.422059e-28
CA[0.625] -0.007008  3.693139e-31  0.199606 -2.364246e-27
CA[0.75]  -0.005004  1.488148e-32  0.145919 -1.480193e-27
CA[0.875] -0.003518  1.730198e-32  0.104471 -9.269836e-28
CA[1]     -0.002444 -1.067417e-31  0.073616 -5.822324e-28
CB[0.125]  0.008045 -3.118432e-03 -0.175326  2.914398e-01
CB[0.25]   0.003920 -3.425395e-03 -0.134072  3.554464e-01
CB[0.375]  0.000041 -3.725247e-03 -0.057591  4.164697e-01
CB[0.5]   -0.003019 -3.981606e-03  0.017618  4.692900e-01
CB[0.625] -0.005168 -4.178727e-03  0.077894  5.117423e-01
CB[0.75]  -0.006515 -4.312737e-03  0.120383  5.434115e-01
CB[0.875] -0.007230 -4.386288e-03  0.146823  5.648378e-01
CB[1]     -0.007482 -4.405336e-03  0.160470  5.770391e-01
CC[0]      0.0

In [20]:
print(fim)

          A1        A2          E1          E2
A1  0.240883  0.045757   -5.529698   -5.576035
A2  0.045757  0.025179   -0.937156   -3.009595
E1 -5.529698 -0.937156  129.021368  114.936210
E2 -5.576035 -3.009595  114.936210  362.830317
